In [1]:
! pip install apache-beam --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.

# Batch

In [2]:
import apache_beam as beam

## ParDo

In [3]:
class C(beam.DoFn):
  def process(self, e):
    return [["1", "2"]]

with beam.Pipeline() as pipeline:
  (
      pipeline
   | "lire les lignes" >> beam.Create([1, 2, 3])
   | "remplacer le texte par 1" >> beam.ParDo(C())
   | "écrire la sortie dans un fichier" >> beam.io.WriteToText("output.txt")
  )

In [4]:
!ls

output.txt-00000-of-00001  sample_data


In [5]:
!cat output.txt-00000-of-00001

['1', '2']
['1', '2']
['1', '2']


### Side outputs

In [6]:
from apache_beam import pvalue


class PairOuImpair(beam.DoFn):
  def process(self, element):
    if element < 5:
      yield pvalue.TaggedOutput("pairs", element)
    else:
      yield pvalue.TaggedOutput("impairs", element)

with beam.Pipeline() as pipeline:
  values = (
      pipeline
      | 'Create produce' >> beam.Create([
          1, 2, 5, 6, 7
      ])
      | 'Pairs et impairs' >> beam.ParDo(PairOuImpair()).with_outputs()
  )
  values.pairs | "print pairs" >> beam.Map(print)
  print("---")
  values.impairs | "print impairs" >> beam.Map(print)

---
1
2
5
6
7


## CombineGlobally

In [7]:
def get_common_items(sets):
  # set.intersection() takes multiple sets as separete arguments.
  # We unpack the `sets` list into multiple arguments with the * operator.
  # The combine transform might give us an empty list of `sets`,
  # so we use a list with an empty set as a default value.
  return set.intersection(*(sets or [set()]))

with beam.Pipeline() as pipeline:
  common_items = (
      pipeline
      | 'Create produce' >> beam.Create([
          {'🍓', '🥕', '🍌', '🍅', '🌶️'},
          {'🍇', '🥕', '🥝', '🍅', '🥔'},
          {'🍉', '🥕', '🍆', '🍅', '🍍'},
          {'🥑', '🥕', '🌽', '🍅', '🥥'},
      ])
      | 'Get common items' >> beam.CombineGlobally(get_common_items)
      | beam.Map(print))


{'🥕', '🍅'}


In [8]:
class C(beam.CombineFn):
  def create_accumulator(self):
    return 0

  def add_input(self, acc, input):
    return acc + input

  def merge_accumulators(self, accumulators):
    r = 0
    for a in accumulators:
      r += a
    return r

  def extract_output(self, accumulator):
    return [accumulator, 1]

with beam.Pipeline() as pipeline:
  data = pipeline | "crée des données" >> beam.Create([1, 2, 3])
  (data
   | beam.CombineGlobally(C())
   | "sorstie" >> beam.io.WriteToText("output.txt")
   )

## GroupByKey

In [9]:
import apache_beam as beam

with beam.Pipeline() as pipeline:
  produce_counts = (
      pipeline
      | 'Create produce counts' >> beam.Create([
          ('spring', '🍓'),
          ('spring', '🥕'),
          ('spring', '🍆'),
          ('spring', '🍅'),
          ('summer', '🥕'),
          ('summer', '🍅'),
          ('summer', '🌽'),
          ('fall', '🥕'),
          ('fall', '🍅'),
          ('winter', '🍆'),
      ])
      | 'Group counts per produce' >> beam.GroupByKey()
      | beam.MapTuple(lambda k, vs: (k, sorted(vs)))  # sort and format
      | beam.Map(print))

('spring', ['🍅', '🍆', '🍓', '🥕'])
('summer', ['🌽', '🍅', '🥕'])
('fall', ['🍅', '🥕'])
('winter', ['🍆'])


## CoGroupByKey

In [11]:
with beam.Pipeline() as p:
  emails_list = [
      ('amy', 'amy@example.com'),
      ('carl', 'carl@example.com'),
      ('julia', 'julia@example.com'),
      ('carl', 'carl@email.com'),
  ]
  phones_list = [
      ('amy', '111-222-3333'),
      ('james', '222-333-4444'),
      ('amy', '333-444-5555'),
      ('carl', '444-555-6666'),
  ]

  emails = p | 'CreateEmails' >> beam.Create(emails_list)
  phones = p | 'CreatePhones' >> beam.Create(phones_list)

  # The result PCollection contains one key-value element for each key in the
  # input PCollections. The key of the pair will be the key from the input and
  # the value will be a dictionary with two entries: 'emails' - an iterable of
  # all values for the current key in the emails PCollection and 'phones': an
  # iterable of all values for the current key in the phones PCollection.
  results = ({'emails': emails, 'phones': phones} | beam.CoGroupByKey())

  results | beam.Map(print)

('amy', {'emails': ['amy@example.com'], 'phones': ['111-222-3333', '333-444-5555']})
('carl', {'emails': ['carl@example.com', 'carl@email.com'], 'phones': ['444-555-6666']})
('julia', {'emails': ['julia@example.com'], 'phones': []})
('james', {'emails': [], 'phones': ['222-333-4444']})


# Streaming

In [12]:
import time


def to_unix_time(time_str: str, time_format='%Y-%m-%d %H:%M:%S') -> int:
  """Converts a time string into Unix time."""
  time_tuple = time.strptime(time_str, time_format)
  return int(time.mktime(time_tuple))

In [ ]:
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.test_stream import TestStream
from apache_beam.transforms.trigger import (AfterWatermark,
                                            AfterProcessingTime,
                                            AfterCount,
                                            AccumulationMode)
from apache_beam.options.pipeline_options import (PipelineOptions,
                                                  StandardOptions,
                                                  TypeOptions)

class GetTimestamp(beam.DoFn):
  def process(self, element, timestamp=beam.DoFn.TimestampParam):
    yield '{} - {}'.format(timestamp.to_utc_datetime(), element)

options = PipelineOptions()
options.view_as(StandardOptions).streaming = True
options.view_as(TypeOptions).allow_unsafe_triggers = True

with TestPipeline(options=options) as pipeline:
  stream = (
      pipeline
      | TestStream()\
        .advance_processing_time(
            advance_by=to_unix_time("2022-01-12 10:30:00")  # on traite les données à 10h30
        ).advance_watermark_to(
            new_watermark=to_unix_time("2022-01-12 10:20:00")  # on considère que toutes les données avant 10h20 ont été traitées
        ).add_elements(
            elements=[
                      ("a", 1),
                      ("b", 2),
                      ("b", 4)
            ],
            event_timestamp=to_unix_time("2022-01-12 10:00:30")  # des données qui sont produites à 10h00:30 (donc déjà traitées)
            #event_timestamp=to_unix_time("2022-01-12 10:20:30")  # des données qui sont produites à 10h20:30 (donc pas déjà traitées)
        ).add_elements(
            elements=[
                      ("a", 3),
                      ("b", 7)
            ],
            event_timestamp=to_unix_time("2022-01-12 10:25:50")  # des données plus récentes que le watermark donc pas encore traitées
        )
  )

  (stream
   | "Windowing" >> beam.WindowInto(
       beam.window.FixedWindows(1*60),  # elles seront aggrégées toutes les minutes
   )
   | "combine" >> beam.CombinePerKey(sum)
   | "Get timestamp" >> beam.ParDo(GetTimestamp())
   | "Print" >> beam.Map(print)
  )

2022-01-12 10:25:59.999999 - ('a', 3)
2022-01-12 10:25:59.999999 - ('b', 7)


In [14]:
with TestPipeline(options=options) as pipeline:
  stream = (
      pipeline
      | TestStream()\
        .advance_processing_time(
            advance_by=to_unix_time("2022-01-12 10:30:00")  # on traite les données à 10h30
        ).advance_watermark_to(
            new_watermark=to_unix_time("2022-01-12 10:20:00")  # on considère que toutes les données avant 10h20 ont été traitées
        ).add_elements(
            elements=[
                      ("a", 1),
                      ("b", 2),
                      ("b", 4)
            ],
            #event_timestamp=to_unix_time("2022-01-12 10:00:30")  # des données qui sont produites à 10h00:30 (donc déjà traitées)
            event_timestamp=to_unix_time("2022-01-12 10:20:30")  # des données qui sont produites à 10h20:30 (donc pas déjà traitées)
        ).add_elements(
            elements=[
                      ("a", 3),
                      ("b", 7)
            ],
            event_timestamp=to_unix_time("2022-01-12 10:25:50")  # des données plus récentes que le watermark donc pas encore traitées
        )
  )

  (stream
   | "Windowing" >> beam.WindowInto(
       beam.window.FixedWindows(1*60),  # elles seront aggrégées toutes les minutes
   )
   | "combine" >> beam.CombinePerKey(sum)
   | "Get timestamp" >> beam.ParDo(GetTimestamp())
   | "Print" >> beam.Map(print)
  )

2022-01-12 10:20:59.999999 - ('a', 1)
2022-01-12 10:20:59.999999 - ('b', 6)
2022-01-12 10:25:59.999999 - ('a', 3)
2022-01-12 10:25:59.999999 - ('b', 7)
